# Super-resolution (SR) with pretrained models: see it, look inside, measure it

Goal: understand what a super-resolution CNN does to **small handwriting** before we train our own.
Everything here uses **pretrained** models in `models/superres/`: generic ESPCN, FSRCNN, EDSR and Real-ESRGAN, plus PaddleOCR's text-specific **Text Gestalt** and **Text Telescope** models.

| Part | What you will see | Needs PaddleOCR? |
|---|---|---|
| **A. See it** | One line of handwriting shrunk to 20 px, then enlarged back by generic and text-specific SR models | recogniser only |
| **B. Look inside ESPCN** | The feature maps after each layer, and how *pixel shuffle* turns 9 channels into a 3x bigger image | no |
| **C. Shrink-and-restore test** | Clear lines from our close-up photos, shrunk to 8-32 px, enlarged back by each method, read by the recogniser: **CER vs text height** | detector + recogniser |
| **D. Real far-away photo** | The same methods on a photo where the writing really is tiny (`cnn_30.png`) | detector + recogniser |

Key idea (week 10 slides): SR = a **fully convolutional** CNN (segmentation slide 15) + an **up-sampling** step (slide 29, here *pixel shuffle*) + **reconstruction training** on images we damaged ourselves (generative slides 13 and 17: inpainting / autoencoder MSE loss). Real-ESRGAN adds adversarial training; Text Gestalt adds stroke supervision; Text Telescope adds character position/content supervision.

What counts as "better" here is **the recogniser reads more characters correctly (lower CER)**, not "the picture looks sharper".

Kernel: **Python 3.12 (whiteboard-ocr)**. Needs `opencv-contrib-python` 4.x as pinned in `requirements.txt` (in our test, OpenCV 5.0 failed to load these `.pb` models).

## 0a. Install the packages this notebook needs

Checks every package this notebook imports and installs **only the missing ones**, into the Python of the **kernel you are running** (`sys.executable`), at the versions pinned in `requirements.txt`. Packages that are already installed are left alone, so a working environment is not changed. Safe to re-run.

In [1]:
import importlib
import importlib.util
import re
import subprocess
import sys
from pathlib import Path

# import name -> pip package name (the version comes from requirements.txt when it is pinned there)
NEEDED = {
    "cv2": "opencv-contrib-python",     # "contrib" build: includes cv2.dnn_superres (ESPCN / FSRCNN / EDSR)
    "numpy": "numpy",
    "pandas": "pandas",
    "openpyxl": "openpyxl",             # pandas needs it to read the .xlsx label files
    "matplotlib": "matplotlib",
    "jiwer": "jiwer",                   # character error rate
    "PIL": "pillow",
    "paddle": "paddlepaddle",
    "paddleocr": "paddleocr",
    "torch": "torch",                 # Real-ESRGAN runtime
    "torchvision": "torchvision",
    "basicsr": "basicsr",
    "realesrgan": "realesrgan",
}

req_file = next((p / "requirements.txt" for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), None)
pins = {}
if req_file:
    for line in req_file.read_text().splitlines():
        m = re.match(r"\s*([A-Za-z0-9_.\-]+)\s*==\s*([^\s#]+)", line)
        if m:
            pins[m.group(1).lower()] = m.group(2)


def installed(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return module in sys.modules


def has_dnn_superres():
    # checked in a separate process, so this kernel does not import cv2 before a possible reinstall
    code = "import cv2; cv2.dnn_superres.DnnSuperResImpl_create()"
    return subprocess.run([sys.executable, "-c", code], capture_output=True).returncode == 0


missing = [pkg for mod, pkg in NEEDED.items() if not installed(mod)]
if "opencv-contrib-python" not in missing and not has_dnn_superres():
    missing.append("opencv-contrib-python")          # cv2 is there but without dnn_superres (plain build or OpenCV 5)
to_install = [f"{p}=={pins[p.lower()]}" if p.lower() in pins else p for p in missing]

print("kernel Python:", sys.executable)
if to_install:
    print("installing:", " ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install])
    importlib.invalidate_caches()
    print("done. If an import below still fails: Kernel > Restart, then Run All.")
else:
    print("all required packages are already installed")

kernel Python: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python
all required packages are already installed


## 0b. Setup

In [2]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import sys
import time
import types
from pathlib import Path

import cv2
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import paddle
import pandas as pd
import torchvision.transforms.functional as tvf

# BasicSR 1.4.2 imports this old torchvision module name. New torchvision
# releases moved the function, so expose the old name before importing BasicSR.
_tv_compat = types.ModuleType("torchvision.transforms.functional_tensor")
_tv_compat.rgb_to_grayscale = tvf.rgb_to_grayscale
sys.modules.setdefault("torchvision.transforms.functional_tensor", _tv_compat)
from basicsr.archs.rrdbnet_arch import RRDBNet
from paddleocr import TextDetection, TextRecognition
from realesrgan import RealESRGANer


def find_repo_root(start=None):
    # the repo root is the folder that contains both requirements.txt and notebooks/
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")


REPO = find_repo_root()
DATA = REPO / "data" / "board2notes_v1"                    # our annotated photos (see data/README.md)
IMG_DIR = DATA / "images"
LABELS = DATA / "board2notes_labels_v1_textonly.xlsx"      # English-only ground truth (formula lines removed)
SR_DIR = REPO / "models" / "superres"                      # pretrained SR weights (see models/README.md)
REALESRGAN_WEIGHTS = SR_DIR / "RealESRGAN_x4plus.pth"
PADDLEOCR_SR_SOURCE = SR_DIR.resolve().parent / "PaddleOCR-text-sr"
TEXT_GESTALT_WEIGHTS = SR_DIR / "text_gestalt" / "sr_tsrn_transformer_strock_train" / "best_accuracy.pdparams"
TEXT_TELESCOPE_WEIGHTS = SR_DIR / "text_telescope" / "Telescope_train" / "best_accuracy.pdparams"
OUT_DIR = REPO / "outputs" / "super_resolution"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- settings ----
TARGET_H = 64          # enlarge a line to about this height before reading (same as notebook 04)
MAX_SCALE = 4.0        # never enlarge more than x4 (same as notebook 04)
METHODS = ["none", "bicubic", "bicubic+sharpen", "espcn", "fsrcnn", "edsr", "realesrgan",
           "text_gestalt", "text_telescope"]
# Real-ESRGAN, Text Gestalt and Text Telescope are relatively slow on CPU; remove them for a quick run.
HEIGHTS = [16, 20, 24, 28, 32]                             # line-BOX heights (px) tested in part C
HUMAN_MIN_H = 20       # box height below which people also struggle to read: set it after looking at part C1
MAX_LINES = 80                                             # lines used in part C (more = slower, steadier numbers)
SEED = 0
DET_ARGS = dict(thresh=0.20, box_thresh=0.40, unclip_ratio=2.00)   # same detector settings as notebook 04
# part C: close-up photos that supply clear reference lines, and how clear a line must be to count
REF_PHOTOS = ["lecture_01.png", "lecture_02.png", "television_03.png", "television_10.png", "quiz_01.png", "quiz_02.png"]
MIN_REF_H = 36          # px: only lines at least this tall in the photo
MAX_REF_CER = 0.15      # only lines the recogniser already reads (almost) perfectly at full size

assert hasattr(cv2, "dnn_superres") and hasattr(cv2.dnn_superres, "DnnSuperResImpl_create"), \
    "needs opencv-contrib-python 4.x (pip install opencv-contrib-python==4.10.0.84)"
for f in ["ESPCN_x2.pb", "ESPCN_x3.pb", "ESPCN_x4.pb"]:
    assert (SR_DIR / f).exists(), f"missing {SR_DIR / f}: see models/README.md"
if "realesrgan" in METHODS:
    assert REALESRGAN_WEIGHTS.exists(), f"missing {REALESRGAN_WEIGHTS}: see models/README.md"
if any(m in METHODS for m in ("text_gestalt", "text_telescope")):
    assert PADDLEOCR_SR_SOURCE.is_dir(), f"missing {PADDLEOCR_SR_SOURCE}: see models/README.md"
    assert TEXT_GESTALT_WEIGHTS.exists(), f"missing {TEXT_GESTALT_WEIGHTS}: see models/README.md"
    assert TEXT_TELESCOPE_WEIGHTS.exists(), f"missing {TEXT_TELESCOPE_WEIGHTS}: see models/README.md"
    sys.path.insert(0, str(PADDLEOCR_SR_SOURCE))
    from ppocr.modeling.architectures import build_model as build_paddleocr_sr_model

labels = pd.read_excel(LABELS, sheet_name="Line Labels", keep_default_na=False)
images = pd.read_excel(LABELS, sheet_name="Images", keep_default_na=False)


def gt_lines(name):
    # ground-truth lines visible in one photo
    return labels.loc[labels.image_id == name, "ground_truth"].tolist()


det_model = TextDetection(model_name="PP-OCRv5_mobile_det")     # finds text boxes
rec_model = TextRecognition(model_name="PP-OCRv6_medium_rec")   # reads one text line per image
print(f"{len(images)} photos, {len(labels)} ground-truth lines, OpenCV {cv2.__version__}")

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv5_mobile_det`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


108 photos, 2733 ground-truth lines, OpenCV 4.10.0


### Helpers: load, shrink (the 'far away' simulation), enlarge (bicubic or SR), read, score

In [3]:
def load(name):
    img = cv2.imread(str(IMG_DIR / name))
    assert img is not None, f"cannot read {IMG_DIR / name}"
    return img


def degrade(crop, h, realistic=True):
    """Make a clear line look like it was photographed from far away: `h` px tall.
    realistic=True also adds the camera's blur and JPEG compression, not only a resize."""
    s = h / crop.shape[0]
    w = max(1, int(round(crop.shape[1] * s)))
    img = crop
    if realistic:
        img = cv2.GaussianBlur(img, (0, 0), sigmaX=max(0.3, 0.5 / s))          # lens / focus blur (in the big image)
    small = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    if realistic:
        ok, buf = cv2.imencode(".jpg", small, [cv2.IMWRITE_JPEG_QUALITY, 70])   # phone JPEG compression
        small = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return small


_sr_cache = {}
_realesrgan_cache = None
_text_sr_cache = {}


def _sr(name, k):
    # pretrained OpenCV dnn_superres model (ESPCN / FSRCNN / EDSR), scale x2 / x3 / x4
    key = (name, k)
    if key not in _sr_cache:
        m = cv2.dnn_superres.DnnSuperResImpl_create()
        m.readModel(str(SR_DIR / f"{name.upper()}_x{k}.pb"))
        m.setModel(name.lower(), k)
        _sr_cache[key] = m
    return _sr_cache[key]


def _realesrgan():
    # Official Real-ESRGAN x4plus: RRDB generator trained with adversarial and perceptual losses.
    # tile=0 processes each small line crop in one pass; half=False is required on CPU.
    global _realesrgan_cache
    if _realesrgan_cache is None:
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        _realesrgan_cache = RealESRGANer(
            scale=4, model_path=str(REALESRGAN_WEIGHTS), model=net,
            tile=0, tile_pad=10, pre_pad=0, half=False, gpu_id=None,
        )
    return _realesrgan_cache


def _text_sr_model(name):
    # Official PaddleOCR text-SR research models. Both expect a 16x64 RGB word crop and output 32x128.
    if name not in _text_sr_cache:
        if name == "text_gestalt":
            cfg = dict(model_type="sr", algorithm="Gestalt",
                       Transform=dict(name="TSRN", STN=True, infer_mode=True))
            weights = TEXT_GESTALT_WEIGHTS
        elif name == "text_telescope":
            cfg = dict(model_type="sr", algorithm="Telescope",
                       Transform=dict(name="TBSRN", STN=True, infer_mode=True))
            weights = TEXT_TELESCOPE_WEIGHTS
        else:
            raise ValueError(name)
        model = build_paddleocr_sr_model(cfg)
        model.set_state_dict(paddle.load(str(weights)))
        model.eval()
        _text_sr_cache[name] = model
    return _text_sr_cache[name]


def _text_sr_enlarge(img, size, method):
    # Follow PaddleOCR's official inference preprocessing. The fixed 4:1 word shape is an
    # important limitation for our longer handwriting lines and is evaluated, not hidden.
    x = cv2.resize(img, (64, 16), interpolation=cv2.INTER_AREA)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype("float32") / 255.0
    with paddle.no_grad():
        y = _text_sr_model(method)(paddle.to_tensor(x))["sr_img"][0].numpy().transpose(1, 2, 0)
    y = cv2.cvtColor(np.clip(y * 255, 0, 255).astype("uint8"), cv2.COLOR_RGB2BGR)
    return cv2.resize(y, size, interpolation=cv2.INTER_CUBIC)


def enlarge(img, s, method):
    """Enlarge by factor s with interpolation or a pretrained SR model."""
    if s <= 1.05 or method == "none":
        return img
    size = (int(round(img.shape[1] * s)), int(round(img.shape[0] * s)))
    if method == "nearest":
        return cv2.resize(img, size, interpolation=cv2.INTER_NEAREST)
    if method == "bicubic":
        return cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
    if method == "bicubic+sharpen":
        up = cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
        blur = cv2.GaussianBlur(up, (0, 0), sigmaX=0.8 * s)
        return cv2.addWeighted(up, 1.6, blur, -0.6, 0)          # unsharp mask (the current pipeline's method)
    if method == "realesrgan":
        # The network reconstructs x4 internally; outscale gives the exact requested size.
        up, _ = _realesrgan().enhance(img, outscale=float(s))
        return up
    if method in ("text_gestalt", "text_telescope"):
        return _text_sr_enlarge(img, size, method)
    # learned SR: smallest model scale >= s (x2..x4), then an exact resize to the target size
    k = int(min(4, max(2, np.ceil(s))))
    up = _sr(method, k).upsample(img)
    return cv2.resize(up, size, interpolation=cv2.INTER_AREA if k > s else cv2.INTER_CUBIC)


def read(imgs):
    # PaddleOCR recogniser on a list of line images -> [(text, confidence)]
    if not imgs:
        return []
    return [(r["rec_text"], float(r["rec_score"])) for r in rec_model.predict(list(imgs))]


def cer(gt, text):
    # character error rate, case-insensitive, capped at 1 (0 = perfect)
    return min(jiwer.cer(gt.lower(), text.lower() or " "), 1.0)


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## A. See it: one line, shrunk and enlarged back

We take one line from a close, sharp photo (`lecture_02.png`), **shrink it to 20 px tall** as if it had been photographed from far away, then enlarge it back **x4** with each method.

- `nearest` just copies each small pixel into a 4x4 block: it shows how little information is left (15 of every 16 output pixels have to be invented).
- `bicubic` invents them by smooth averaging (blurry).
- `espcn` / `fsrcnn` / `edsr` invent them with a CNN that learned "what sharp images look like" from training photos.

Under each picture: what the PaddleOCR recogniser reads, and its CER against the ground truth.

In [4]:
DEMO_PHOTO = "lecture_02.png"
DEMO_BOX = (1985, 1290, 2480, 1370)        # x0, y0, x1, y1 of one line in that photo
DEMO_GT = "Stochastic Gradient Descent"
LR_H = 20                                  # "far away" height of the line

x0, y0, x1, y1 = DEMO_BOX
hr = load(DEMO_PHOTO)[y0:y1, x0:x1]        # HR = high resolution: the clear original
lr = degrade(hr, LR_H)                     # LR = low resolution: what a far photo would give
s = hr.shape[0] / lr.shape[0]              # enlarge back to the original size, so all pictures are comparable

panels = [("original (close-up photo)", hr, None), (f"shrunk to {LR_H} px: the input", lr, None)]
for m in ["nearest", "bicubic", "bicubic+sharpen", "espcn", "fsrcnn", "edsr", "realesrgan",
          "text_gestalt", "text_telescope"]:
    if m not in METHODS and m != "nearest":
        continue
    t0 = time.time()
    up = enlarge(lr, s, m)
    panels.append((m, up, time.time() - t0))
texts = read([p[1] for p in panels])

fig, axes = plt.subplots(len(panels), 1, figsize=(10, 1.25 * len(panels)))
for ax, (name, im, t), (txt, conf) in zip(axes, panels, texts):
    ax.imshow(rgb(im), interpolation="nearest")
    timing = f", {1000 * t:.0f} ms" if t is not None else ""
    ax.set_title(f"{name} ({im.shape[1]}x{im.shape[0]} px{timing})  ->  reads '{txt}'  "
                 f"(CER {cer(DEMO_GT, txt):.2f}, conf {conf:.2f})", fontsize=9, loc="left")
    ax.axis("off")
plt.tight_layout(); plt.show()

# zoom on the first letters: here you can see the 4x4 blocks of `nearest` and what each method invents
zw = hr.shape[1] // 4
zoom = [p for p in panels if p[0] in ("original (close-up photo)", "nearest", "bicubic", "bicubic+sharpen",
                                        "espcn", "edsr", "realesrgan", "text_gestalt", "text_telescope")]
fig, axes = plt.subplots(1, len(zoom), figsize=(3 * len(zoom), 2.2))
for ax, (name, im, _) in zip(axes, zoom):
    ax.imshow(rgb(im[:, :zw]), interpolation="nearest"); ax.set_title(name.split(" (")[0], fontsize=9); ax.axis("off")
fig.suptitle("zoom: left quarter of the line"); plt.tight_layout(); plt.show()

<Figure size 1000x1125 with 9 Axes>

<Figure size 2100x220 with 7 Axes>

### A2. The same thing as numbers: brightness along one row of pixels

A horizontal slice through the middle of the line. Ink is dark (low values), the board is bright (~255).
A sharp stroke is a **narrow, deep dip**. Bicubic gives **wide, shallow dips** (blur). Look at whether the SR curves get the dips narrower and deeper, closer to the original.

In [5]:
row = hr.shape[0] // 2
gray = lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).astype(float)
cols = slice(0, min(160, hr.shape[1]))     # first 160 px of the row, so single strokes are visible

plt.figure(figsize=(12, 3.5))
plt.plot(gray(hr)[row, cols], "k", lw=2.5, label="original (target)")
for name, im, _ in panels[2:]:
    if name in ("bicubic", "espcn", "edsr", "realesrgan", "text_gestalt", "text_telescope"):
        plt.plot(gray(im)[row, cols], lw=1.3, label=name)
plt.xlabel("pixel column"); plt.ylabel("brightness (0 = black ink, 255 = white board)")
plt.title(f"row {row} of the demo line"); plt.legend(); plt.tight_layout(); plt.show()

<Figure size 1200x350 with 1 Axes>

## B. Look inside ESPCN

To see the layers we load ESPCN's weights straight from the `.pb` file and run the network **ourselves in NumPy**, one layer at a time (OpenCV's `upsample` only returns the final image).
The cell at the end checks our NumPy version gives the same output as OpenCV's.

ESPCN x3 (Shi et al., 2016), as stored in `ESPCN_x3.pb`:

```
input (Y = brightness channel)  1  x H x W
conv1 5x5 + ReLU            ->  64 x H x W     feature maps: the network's "working notes"
conv2 3x3 + ReLU            ->  32 x H x W
conv3 3x3                   ->   9 x H x W     9 = 3 x 3: one channel per output sub-pixel
pixel shuffle + tanh        ->   1 x 3H x 3W   the enlarged image
```
No flatten, no fully-connected layer: the output stays a grid (a *fully convolutional* network).

In [6]:
def _varint(b, i):
    r = s = 0
    while True:
        x = b[i]; i += 1
        r |= (x & 0x7F) << s; s += 7
        if x < 0x80:
            return r, i


def _fields(b):
    # tiny protobuf reader (a .pb file is a protobuf message): yields (field number, wire type, value)
    i = 0
    while i < len(b):
        key, i = _varint(b, i); f, w = key >> 3, key & 7
        if w == 0:
            v, i = _varint(b, i)
        elif w == 1:
            v = b[i:i + 8]; i += 8
        elif w == 2:
            n, i = _varint(b, i); v = b[i:i + n]; i += n
        elif w == 5:
            v = b[i:i + 4]; i += 4
        else:
            raise ValueError(f"unsupported wire type {w}")
        yield f, w, v


def read_tf_weights(pb_path):
    """{name: array} for every constant (weights, biases) in a frozen TensorFlow graph, without TensorFlow."""
    out = {}
    for f, _, node in _fields(Path(pb_path).read_bytes()):
        if f != 1:                                    # GraphDef.node
            continue
        name = op = tensor = None
        for nf, _, nv in _fields(node):
            if nf == 1:
                name = nv.decode()
            elif nf == 2:
                op = nv.decode()
            elif nf == 5:                             # attribute map entry
                key = val = None
                for ef, _, ev in _fields(nv):
                    if ef == 1:
                        key = ev.decode()
                    elif ef == 2:
                        val = ev
                if key == "value" and val is not None:
                    tensor = next((av for af, _, av in _fields(val) if af == 8), None)
        if op != "Const" or tensor is None:
            continue
        shape, content, floats = [], b"", []
        for tf_, tw, tv in _fields(tensor):
            if tf_ == 2:                              # shape
                for sf, _, sv in _fields(tv):
                    if sf == 2:
                        shape += [dv for df, _, dv in _fields(sv) if df == 1]
            elif tf_ == 4:                            # raw float32 bytes
                content = tv
            elif tf_ == 5:                            # float values stored one by one
                floats += list(np.frombuffer(tv, "<f4")) if tw == 2 else [np.frombuffer(tv, "<f4")[0]]
        arr = np.frombuffer(content, "<f4") if content else np.array(floats, np.float32)
        if shape and arr.size == 1 and np.prod(shape) > 1:
            arr = np.full(int(np.prod(shape)), arr[0], np.float32)
        out[name] = arr.reshape(shape) if shape else arr
    return out


def conv2d_same(x, w, b):
    # x: (C_in, H, W); w: TensorFlow layout (kh, kw, C_in, C_out). Zero padding keeps H x W ("SAME")
    y = np.zeros((w.shape[3], *x.shape[1:]), np.float32)
    for o in range(w.shape[3]):
        for i in range(w.shape[2]):
            y[o] += cv2.filter2D(x[i], -1, w[:, :, i, o], borderType=cv2.BORDER_CONSTANT)
        y[o] += b[o]
    return y


def pixel_shuffle(x, r):
    # (r*r, H, W) -> (r*H, r*W): channel i*r + j becomes sub-pixel (i, j) of every r x r block
    _, h, w = x.shape
    return x.reshape(r, r, h, w).transpose(2, 0, 3, 1).reshape(h * r, w * r)


R = 3
W = read_tf_weights(SR_DIR / f"ESPCN_x{R}.pb")
for k in ["f1", "b1", "f2", "b2", "f3", "b3"]:
    print(f"{k}: {W[k].shape}")
print("total weights:", sum(W[k].size for k in ["f1", "b1", "f2", "b2", "f3", "b3"]))

f1: (5, 5, 1, 64)
b1: (64,)
f2: (3, 3, 64, 32)
b2: (32,)
f3: (3, 3, 32, 9)
b3: (9,)
total weights: 22729


In [7]:
# run ESPCN on the 20 px demo line, layer by layer (on brightness Y, like OpenCV does; colour is added back by bicubic)
y_lr = cv2.cvtColor(lr, cv2.COLOR_BGR2YCrCb)[:, :, 0].astype(np.float32) / 255
f1 = np.maximum(conv2d_same(y_lr[None], W["f1"], W["b1"]), 0)    # 64 maps
f2 = np.maximum(conv2d_same(f1, W["f2"], W["b2"]), 0)            # 32 maps
f3 = conv2d_same(f2, W["f3"], W["b3"])                           # 9 maps
y_sr = np.tanh(pixel_shuffle(f3, R))                             # 1 image, 3x bigger


def grid(maps, title, n, cols=4):
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 0.9 * rows + 0.6))
    for i, ax in enumerate(np.ravel(axes)):
        ax.axis("off")
        if i < n:
            ax.imshow(maps[i], cmap="gray"); ax.set_title(f"channel {i}", fontsize=8)
    fig.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()


plt.figure(figsize=(8, 1.2)); plt.imshow(y_lr, cmap="gray"); plt.axis("off")
plt.title(f"input: {y_lr.shape[1]}x{y_lr.shape[0]} px (brightness only)", fontsize=10); plt.show()
grid(f1, "conv1: 8 of 64 feature maps (edge / stroke detectors; hard to interpret). All-black = that filter found nothing here", 8)
grid(f2, "conv2: 8 of 32 feature maps (combinations; even harder to interpret)", 8)
grid(f3, "conv3: all 9 maps. Training forced these to be PIXEL VALUES: each is the image seen from a slightly different\n"
     "sub-pixel position (top-left, top-middle, ... of each 3x3 block), so they look almost the same", 9, cols=3)

<Figure size 800x120 with 1 Axes>

<Figure size 1400x240 with 8 Axes>

<Figure size 1400x240 with 8 Axes>

<Figure size 1400x330 with 9 Axes>

### B2. Pixel shuffle, with real numbers

Pick one small (LR) pixel that sits on ink. The last layer gave it **9 numbers**. Pixel shuffle lays those 9 numbers out as the **3x3 block** that this one pixel becomes in the enlarged image. Nothing is averaged: the network predicted every output pixel directly.

In [8]:
yy, xx = np.unravel_index(np.argmin(y_lr[2:-2, 2:-2]), np.array(y_lr[2:-2, 2:-2]).shape)   # darkest pixel = on ink
yy, xx = yy + 2, xx + 2
nine = np.tanh(f3[:, yy, xx]) * 255
print(f"small pixel at row {yy}, col {xx}: brightness {y_lr[yy, xx] * 255:.0f}")
print("its 9 output values (channel 0..8):", np.round(nine).astype(int).tolist())
print("\nafter pixel shuffle, the 3x3 block at rows", 3 * yy, "-", 3 * yy + 2, ", cols", 3 * xx, "-", 3 * xx + 2, ":")
for row_vals in np.round(y_sr[3 * yy:3 * yy + 3, 3 * xx:3 * xx + 3] * 255).astype(int).tolist():
    print("   ", row_vals)

fig, ax = plt.subplots(1, 2, figsize=(14, 2.2))
ax[0].imshow(y_lr, cmap="gray", vmin=0, vmax=1); ax[0].set_title(f"input {y_lr.shape[1]}x{y_lr.shape[0]}")
ax[0].plot(xx, yy, "r+", ms=12)
ax[1].imshow(np.clip(y_sr, 0, 1), cmap="gray", vmin=0, vmax=1); ax[1].set_title(f"after pixel shuffle {y_sr.shape[1]}x{y_sr.shape[0]}")
ax[1].add_patch(plt.Rectangle((3 * xx - 0.5, 3 * yy - 0.5), 3, 3, fill=False, ec="r"))
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

# check: our NumPy ESPCN == OpenCV's ESPCN (compare the brightness channel)
y_cv = cv2.cvtColor(_sr("espcn", R).upsample(lr), cv2.COLOR_BGR2YCrCb)[:, :, 0].astype(np.float32)
diff = np.abs(np.clip(y_sr, 0, 1) * 255 - y_cv)
print(f"\nNumPy vs OpenCV ESPCN: mean |difference| = {diff.mean():.2f} grey levels (max {diff.max():.0f}); "
      "under ~1-2 means the same network")

small pixel at row 7, col 71: brightness 70
its 9 output values (channel 0..8): [101, 91, 99, 86, 71, 85, 92, 85, 94]

after pixel shuffle, the 3x3 block at rows 21 - 23 , cols 213 - 215 :
    [101, 91, 99]
    [86, 71, 85]
    [92, 85, 94]


<Figure size 1400x220 with 2 Axes>


NumPy vs OpenCV ESPCN: mean |difference| = 0.25 grey levels (max 1); under ~1-2 means the same network


## C. Shrink-and-restore test: CER vs line height

1. **Reference lines**: detect lines in close-up photos at full resolution, read each one, and keep it only if it matches a ground-truth line almost perfectly (CER <= 0.15) and is at least 36 px tall. These are lines the recogniser can read when they are big enough.
2. **Shrink** each one so its line box is 16, 20, ... 32 px tall (with blur + JPEG, `degrade`).
3. **Enlarge** to ~64 px with each method (max x4) and **read** it. `none` = give the small line to the recogniser as is.

**Which height?** All heights here are the **line box** drawn by the detector, which is taller than the letters (it adds a margin). The cell below measures how tall the ink actually is inside the boxes, so the plot can show both.
Below about 16 px boxes the letters are only a few pixels tall and **people cannot read them either**, so those sizes are not tested: no method should be expected to read what a person cannot.

In [9]:
def line_height(poly):
    # height of a (roughly horizontal) text-line box: mean of its two short sides (as in notebook 04)
    p = np.asarray(poly, float).reshape(-1, 2)
    if len(p) != 4:
        return float(np.ptp(p[:, 1]))
    return float((np.linalg.norm(p[0] - p[3]) + np.linalg.norm(p[1] - p[2])) / 2)


def quad_crop(img, poly):
    # cut a text line along its own (possibly tilted) box (as in notebook 04)
    p = np.float32(np.asarray(poly, float).reshape(-1, 2))
    if len(p) != 4:
        p = np.float32(cv2.boxPoints(cv2.minAreaRect(p)))
    s_, d_ = p.sum(1), np.diff(p, axis=1).ravel()
    p = np.float32([p[s_.argmin()], p[d_.argmin()], p[s_.argmax()], p[d_.argmax()]])
    w = int(round(max(np.linalg.norm(p[0] - p[1]), np.linalg.norm(p[3] - p[2]))))
    h = int(round(max(np.linalg.norm(p[0] - p[3]), np.linalg.norm(p[1] - p[2]))))
    M = cv2.getPerspectiveTransform(p, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    return cv2.warpPerspective(img, M, (max(w, 1), max(h, 1)), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def detect_full(img):
    # text boxes at FULL resolution (no shrinking), so small writing is not missed
    res = det_model.predict(img, limit_side_len=max(img.shape[:2]), limit_type="max", **DET_ARGS)
    return [np.asarray(p, float).reshape(-1, 2) for p in res[0]["dt_polys"]]


ref, seen = [], set()
for name in REF_PHOTOS:
    t0 = time.time()
    img, gts = load(name), gt_lines(name)
    polys = detect_full(img)
    crops = [quad_crop(img, p) for p in polys]
    reads = read(crops)
    kept = 0
    for crop, (txt, _) in sorted(zip(crops, reads), key=lambda cr: -cr[0].shape[0]):
        if crop.shape[0] < MIN_REF_H or not gts:
            continue
        g = min(gts, key=lambda g: cer(g, txt))
        if cer(g, txt) <= MAX_REF_CER and g not in seen:
            seen.add(g); kept += 1
            ref.append(dict(photo=name, gt=g, crop=crop, height=crop.shape[0], read_full_size=txt))
    print(f"{name}: {len(polys)} boxes, {kept} reference lines kept ({time.time() - t0:.0f} s)")

assert ref, "no reference lines found: check REF_PHOTOS, or relax MIN_REF_H / MAX_REF_CER in the settings cell"


def ink_height(crop):
    # height of the rows that contain ink (letters incl. ascenders / descenders; an underline can add a bit)
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    rows = bw.mean(1) / 255
    on = np.nonzero(rows >= 0.25 * rows.max())[0]
    return int(on[-1] - on[0] + 1) if len(on) else crop.shape[0]


for r in ref:
    r["ink_ratio"] = ink_height(r["crop"]) / r["height"]
rng = np.random.default_rng(SEED)
if len(ref) > MAX_LINES:
    ref = [ref[i] for i in sorted(rng.choice(len(ref), MAX_LINES, replace=False))]
print(f"\n{len(ref)} reference lines, original box heights {min(r['height'] for r in ref)}-{max(r['height'] for r in ref)} px")
INK_RATIO = float(np.median([r["ink_ratio"] for r in ref]))
print(f"the ink fills about {INK_RATIO:.0%} of the box height (median): a 20 px box holds letters about {20 * INK_RATIO:.0f} px tall")

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


lecture_01.png: 214 boxes, 64 reference lines kept (16 s)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


lecture_02.png: 127 boxes, 40 reference lines kept (12 s)


Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.


television_03.png: 5 boxes, 2 reference lines kept (6 s)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


television_10.png: 7 boxes, 0 reference lines kept (7 s)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


quiz_01.png: 11 boxes, 1 reference lines kept (7 s)
quiz_02.png: 15 boxes, 0 reference lines kept (6 s)

80 reference lines, original box heights 43-334 px
the ink fills about 41% of the box height (median): a 20 px box holds letters about 8 px tall


### C1. Can a person read it? (look before you scroll)

Four reference lines at each tested box height, exactly as small as the recogniser would get them (shown bigger on screen, with smooth zoom like a phone). **Try to read them before looking at the answers printed below the figure.**
Then set `HUMAN_MIN_H` in the settings cell to the smallest height you could read without guessing, and re-run the plot cell: the area below it is shaded as "hard for people too".

In [10]:
check = [ref[i] for i in np.random.default_rng(SEED + 1).choice(len(ref), min(4, len(ref)), replace=False)]
fig, axes = plt.subplots(len(HEIGHTS), len(check), figsize=(4.2 * len(check), 1.0 * len(HEIGHTS)), squeeze=False)
for i, h in enumerate(HEIGHTS):
    for j, r in enumerate(check):
        l = degrade(r["crop"], h)
        axes[i, j].imshow(rgb(l), interpolation="bicubic"); axes[i, j].axis("off")
        axes[i, j].set_title(f"box {h} px (letters ~{h * INK_RATIO:.0f} px), line {j + 1}", fontsize=8, loc="left")
plt.tight_layout(); plt.show()
print("answers:", " | ".join(f"line {j + 1}: {r['gt']}" for j, r in enumerate(check)))

<Figure size 1680x500 with 20 Axes>

answers: line 1: Augmentation + Dropout | line 2: add zero around borders | line 3: normalises values close to 0 | line 4: Model


In [11]:
_warm = degrade(ref[0]["crop"], 16)
for m in METHODS:                       # load every model once, so its loading time is not counted below
    enlarge(_warm, 4.0, m)

rows = []
for h in HEIGHTS:
    lrs = [degrade(r["crop"], h) for r in ref]
    for m in METHODS:
        t0 = time.time()
        ins = lrs if m == "none" else [enlarge(l, min(TARGET_H / l.shape[0], MAX_SCALE), m) for l in lrs]
        ms_per_line = 1000 * (time.time() - t0) / len(lrs)
        for r, (txt, conf) in zip(ref, read(ins)):
            rows.append(dict(height=h, method=m, gt=r["gt"], read=txt, conf=conf, cer=cer(r["gt"], txt),
                             exact=txt.strip().lower() == r["gt"].strip().lower(), ms_per_line=ms_per_line))
    print(f"height {h:>2} px done")
bench = pd.DataFrame(rows)
bench.to_csv(OUT_DIR / "shrink_restore_results.csv", index=False)

table = bench.pivot_table(index="method", columns="height", values="cer", aggfunc="mean").loc[METHODS]
print("mean CER (lower is better); rows = enlargement method, columns = line BOX height in px")
display(table.round(3))
speed = bench.groupby("method")["ms_per_line"].mean().loc[METHODS].round(1)
display(speed.to_frame("enlarge time per line (ms)"))

height 16 px done
height 20 px done
height 24 px done
height 28 px done
height 32 px done
mean CER (lower is better); rows = enlargement method, columns = line BOX height in px


height,16,20,24,28,32
method,,,,,
none,0.335,0.132,0.079,0.054,0.037
bicubic,0.346,0.148,0.078,0.057,0.034
bicubic+sharpen,0.389,0.161,0.081,0.054,0.034
espcn,0.400,0.155,0.081,0.059,0.035
fsrcnn,0.415,0.171,0.087,0.061,0.034
edsr,0.394,0.168,0.078,0.059,0.036
realesrgan,0.561,0.283,0.123,0.070,0.042


,enlarge time per line (ms)
method,
none,0.0
bicubic,0.0
bicubic+sharpen,0.1
espcn,0.6
fsrcnn,1.2
edsr,183.3
realesrgan,307.1


In [12]:
fig, ax = plt.subplots(figsize=(9, 5))
if HUMAN_MIN_H > min(HEIGHTS):
    ax.axvspan(min(HEIGHTS) - 1, HUMAN_MIN_H, color="grey", alpha=0.15, label=f"hard for people too (< {HUMAN_MIN_H} px box)")
for m in METHODS:
    ax.plot(table.columns, table.loc[m], marker="o", lw=2.5 if m in ("none", "bicubic+sharpen", "text_gestalt", "text_telescope") else 1.3, label=m)
ax.set_xlabel("line BOX height after shrinking (px)"); ax.set_ylabel("mean CER (lower = better)")
top = ax.secondary_xaxis("top", functions=(lambda x: x * INK_RATIO, lambda x: x / INK_RATIO))
top.set_xlabel(f"about how tall the letters are (px), ink = {INK_RATIO:.0%} of the box")
ax.set_title(f"Shrink-and-restore: {len(ref)} lines from our close-up photos", pad=30)
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "cer_vs_height.png", dpi=150); plt.show()

# gain of each method over the current pipeline method (bicubic+sharpen): positive = fewer errors
gain = (table.loc["bicubic+sharpen"] - table).drop(index="bicubic+sharpen")
print("CER reduction vs bicubic+sharpen (positive = better than the current pipeline)")
display(gain.round(3))

<Figure size 900x500 with 1 Axes>

CER reduction vs bicubic+sharpen (positive = better than the current pipeline)


height,16,20,24,28,32
method,,,,,
none,0.054,0.029,0.002,0.001,-0.003
bicubic,0.043,0.013,0.003,-0.003,-0.001
espcn,-0.011,0.006,0.000,-0.004,-0.002
fsrcnn,-0.026,-0.010,-0.007,-0.007,-0.001
edsr,-0.004,-0.007,0.002,-0.004,-0.003
realesrgan,-0.171,-0.122,-0.042,-0.016,-0.008


### C2. What the recogniser actually sees: one line at each height and method

In [13]:
example = max(ref, key=lambda r: len(r["gt"]) if len(r["gt"]) <= 30 else 0)   # a longish line that fits a panel
show_h = [h for h in HEIGHTS if h in (16, 20, 24, 32)] or HEIGHTS[:4]
show_m = [m for m in ["none", "bicubic+sharpen", "espcn", "realesrgan", "text_gestalt", "text_telescope"] if m in METHODS]
fig, axes = plt.subplots(len(show_h), len(show_m), figsize=(4.2 * len(show_m), 1.1 * len(show_h)), squeeze=False)
for i, h in enumerate(show_h):
    l = degrade(example["crop"], h)
    for j, m in enumerate(show_m):
        im = l if m == "none" else enlarge(l, min(TARGET_H / l.shape[0], MAX_SCALE), m)
        txt, _ = read([im])[0]
        axes[i, j].imshow(rgb(im), interpolation="nearest"); axes[i, j].axis("off")
        axes[i, j].set_title(f"box {h}px {m}: '{txt}' (CER {cer(example['gt'], txt):.2f})", fontsize=8, loc="left")
fig.suptitle(f"ground truth: '{example['gt']}'"); plt.tight_layout(); plt.show()

<Figure size 2100x440 with 20 Axes>

## D. A real far-away photo

Part C *simulates* distance. Here the distance is real: `cnn_30.png` is a whole-room shot where the board's 7 lines are tiny.
Lines are detected at full resolution, enlarged with each method and read; each ground-truth line is matched with its best reading (extra detections, e.g. wall notices, are ignored).

In [14]:
FAR_PHOTO = "cnn_30.png"
far, far_gt = load(FAR_PHOTO), gt_lines(FAR_PHOTO)
far_polys = detect_full(far)
far_crops = [quad_crop(far, p) for p in far_polys]
print(f"{FAR_PHOTO}: {len(far_polys)} boxes, line heights {sorted(c.shape[0] for c in far_crops)} px")

far_rows = []
for m in METHODS:
    ins = far_crops if m == "none" else [enlarge(c, float(np.clip(TARGET_H / c.shape[0], 1, MAX_SCALE)), m) for c in far_crops]
    outs = read(ins)
    for g in far_gt:
        txt = min((o[0] for o in outs), key=lambda t: cer(g, t), default="")
        far_rows.append(dict(method=m, gt=g, read=txt, cer=cer(g, txt)))
far_df = pd.DataFrame(far_rows)
display(far_df.pivot(index="gt", columns="method", values="read")[METHODS])
display(far_df.groupby("method")["cer"].mean().loc[METHODS].round(3).to_frame("mean CER"))

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


cnn_30.png: 29 boxes, line heights [7, 7, 10, 10, 12, 12, 13, 18, 19, 19, 19, 19, 21, 21, 24, 25, 25, 25, 25, 26, 27, 28, 28, 28, 29, 29, 30, 52, 72] px


method,none,bicubic,bicubic+sharpen,espcn,fsrcnn,edsr,realesrgan
gt,,,,,,,
CNN Architecture,CMN Achitecture,CMN Achitecture,CMN Arhitecture,CMN Achitecture,CMN Achitecture,CMN Achitecture,CMN Achitecture
"Dropout, Batch Normalization","Droput, Bath Normoliastion","Dropaut, Bath Normoliestion","Dropaut, Bath Normoliestion","Dropout, Bahn Normoliestion","Droput, Bahn Normaliestion","Dropout, Batch Hermalrenstion","Draput, Batch Normoliss910m"
"LeNet, Alexnet, Resnet","Levet. Akxret, Resnet","Levet. Akxet, Reset","Livet. Akxet, Reset","Levet. Akxret, Resnet","Levet. Akxret, Resnet","Levet. Akxet, Resnet","Levet. Abxnet, Resnet"
Saliency via occlusion,Solancy ria oxlusien,Solancy ria oxlvosin,Solancy ria oxelvsion,Solacy via oxclsion,Solacy ria oxlusion,Solancy via eoxcluosion,Soloancy via oalvsian
Visualise feature layer,Visalire testure layer,Viwalire testure layer,Vialire testure layer,Vialire testure layer,Viawbire testure layer,Vialire testure layer,Vinwalire teature layer
Visualise filter & output,Visvalize filter outpat,Visvalize filter outpt,Visvalize filter outpt,Virvalize filter outpt,Virvalize filter outpt,Vivalise filter outpt,Visvalise filter outpt
Visualising CNN,Visvalising ChN,Visvalising chn,Visvalising chr,Visualising cun,Visvalising chn,Visvalising chn,Visualsing cun


,mean CER
method,
none,0.194
bicubic,0.214
bicubic+sharpen,0.217
espcn,0.189
fsrcnn,0.223
edsr,0.199
realesrgan,0.202


## E. How to read the results (fill in after running)

- **Only the readable zone counts.** Look at heights at or above `HUMAN_MIN_H`: there a person can read the line, so a better method *could* help. Below it, the letters are gone for everyone.
- **Where do the curves separate?** Above ~32 px every method works (no enlarging needed). If the curves stay together between `HUMAN_MIN_H` and 32 px, enlarging (bicubic or SR) does not matter for us.
- **Does any SR model beat `bicubic+sharpen`** (the current pipeline) in that zone, by more than a few hundredths of CER? If not, pretrained SR is not worth its time cost, and that is a result too.
- **Speed**: compare ms per line. The live demo runs on a laptop CPU.
- **Why pretrained SR may disappoint**: these models learned from ordinary photos (landscapes, animals), not marker strokes, and they never saw our blur + JPEG damage.
- **Real-ESRGAN warning**: it can generate plausible-looking high-frequency detail. That can make a line look clearer while subtly changing a character, so judge it by CER and exact-match rate rather than visual sharpness alone.
- **Is it worth deploying?** Real-ESRGAN is much larger and slower than bicubic, ESPCN or FSRCNN on CPU. Only add it to the main pipeline if the OCR improvement is consistent enough to justify that latency.
- **Text-specific models**: Text Gestalt was trained to preserve strokes; Text Telescope was trained with character position/content supervision. Unlike the generic models, both were trained on TextZoom word crops.
- **Shape limitation**: their official preprocessing forces every input to a 16x64 word crop and reconstructs 32x128. Our longer line crops are therefore stretched; if the models are promising, the next experiment should use word boxes or fine-tune a variable-width model.
- **Text too small for anyone**: rather than trying harder, the app can measure the text height (as `smart_ocr` already does) and tell the user "writing too small: step closer or zoom in".
- **Next step (only if the readable zone shows a gap)**: fine-tune the best text-specific model on *our* handwriting and degradation (blur, compression, glare and distance), keep entire photos/writers out of training, and rerun parts C and D.